# 🚀 DermaSense AI - Huấn Luyện Trên Google Colab (GitHub Edition)
**Quy trình chuyên nghiệp:** GitHub → Colab → Google Drive

**⚠️ BẮT BUỘC: Bật GPU trước khi chạy!**
- `Runtime` → `Change runtime type` → `T4 GPU` → `Save`

## Bước 1: Clone mã nguồn từ GitHub
Tự động kéo code mới nhất, không cần zip hay upload thủ công!

In [ ]:
import os

REPO_URL = 'https://github.com/PHANQUOCTHANG/DermaSense_AI.git'
PROJECT_DIR = '/content/DermaSense_AI'

if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}
    !git reset --hard HEAD
    !git pull origin main
    print('Da cap nhat code moi nhat tu GitHub!')
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
    print('Da clone thanh cong tu GitHub!')

!ls

## Bước 2: Tải dữ liệu DermNet từ Kaggle
Bấm Play → Nhập **Kaggle Username** và **API Key** vào 2 ô hiện ra.

**Cách lấy Username và Key:**
1. Vào [kaggle.com/settings](https://www.kaggle.com/settings) → mục **API**
2. Bấm vào **dấu 3 chấm (⋮)** bên cạnh Token đã tạo → chọn **View** hoặc **Copy**
3. Copy **Username** và **Key** rồi dán vào 2 ô bên dưới

In [ ]:
import json, os
from getpass import getpass

kaggle_username = input('Nhap Kaggle Username: ')
kaggle_key = getpass('Nhap Kaggle API Key (an khi go): ')

# Tao file kaggle.json tu dong
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!pip install -q kaggle
print('\nDang tai du lieu DermNet (1.7GB)...')
!kaggle datasets download -d shubhamgoel27/dermnet -p data/raw/dermnet_raw --unzip

print('\nDang tai du lieu HAM10000 (3GB)...')
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/raw/ham10000_raw --unzip

print('\nDang tai du lieu PAD-UFES-20 (1GB)...')
!kaggle datasets download -d wanderdust/pad-ufes-20 -p data/raw/pad_ufes_raw --unzip

print('\nTai xong tat ca 3 tap du lieu!')


## Bước 3: Chuẩn bị dữ liệu (Ingestion)

In [ ]:
import shutil
import random
import pandas as pd
import numpy as np
from pathlib import Path
import os

img_out = Path('data/raw/images')
if img_out.exists():
    shutil.rmtree(img_out)
img_out.mkdir(parents=True, exist_ok=True)

metadata_list = []

print("--- XU LY DERMNET ---")
dermnet_dir = Path('data/raw/dermnet_raw')
d_count = 0
if dermnet_dir.exists():
    for split_dir in ['train', 'test']:
        split_path = dermnet_dir / split_dir
        if split_path.exists():
            for cls_dir in split_path.iterdir():
                if cls_dir.is_dir():
                    for img_path in cls_dir.glob('*.*'):
                        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                            img_id = f'DN_{d_count:07d}'
                            new_path = img_out / f'{img_id}{img_path.suffix}'
                            shutil.copy2(img_path, new_path)
                            
                            metadata_list.append({
                                'image_id': img_id, 'diagnosis': cls_dir.name,
                                'age': 45, 'sex': None, 'anatom_site': None, 
                                'duration': 30, 'symptoms': 'none', 
                                'skin_type': 'III', 'family_history': 'no'
                            })
                            d_count += 1
print(f"Da xu ly {d_count} anh DermNet.")

print("--- XU LY HAM10000 ---")
ham_dir = Path('data/raw/ham10000_raw')
h_count = 0
if ham_dir.exists():
    ham_map = {
        'nv': 'Melanoma and Nevi', 'mel': 'Melanoma and Nevi',
        'bkl': 'Seborrheic Keratoses and Benign Tumors', 'bcc': 'Actinic Keratosis and Skin Cancer',
        'akiec': 'Actinic Keratosis and Skin Cancer', 'vasc': 'Vascular Tumors',
        'df': 'Seborrheic Keratoses and Benign Tumors'
    }
    ham_csv = ham_dir / 'HAM10000_metadata.csv'
    if ham_csv.exists():
        df_ham = pd.read_csv(ham_csv)
        for _, row in df_ham.iterrows():
            img_id_orig = row['image_id']
            img_path = None
            for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
                p = ham_dir / part / f"{img_id_orig}.jpg"
                if p.exists():
                    img_path = p
                    break
            
            if img_path:
                img_id = f'HAM_{h_count:07d}'
                new_path = img_out / f'{img_id}.jpg'
                shutil.copy2(img_path, new_path)
                
                label = ham_map.get(row['dx'], 'Systemic Diseases')
                age = row['age'] if pd.notna(row['age']) else 45
                sex = row['sex'] if pd.notna(row['sex']) and row['sex'] in ['male', 'female'] else None
                loc = row['localization'] if pd.notna(row['localization']) and row['localization'] != 'unknown' else None
                
                metadata_list.append({
                    'image_id': img_id, 'diagnosis': label, 'age': age, 'sex': sex, 'anatom_site': loc, 
                    'duration': 30, 'symptoms': 'none', 'skin_type': 'III', 'family_history': 'no'
                })
                h_count += 1
print(f"Da xu ly {h_count} anh HAM10000.")

print("--- XU LY PAD-UFES-20 ---")
pad_dir = Path('data/raw/pad_ufes_raw')
p_count = 0
if pad_dir.exists():
    pad_map = {
        'BCC': 'Actinic Keratosis and Skin Cancer', 'SCC': 'Actinic Keratosis and Skin Cancer',
        'ACK': 'Actinic Keratosis and Skin Cancer', 'SEK': 'Seborrheic Keratoses and Benign Tumors',
        'BOD': 'Seborrheic Keratoses and Benign Tumors', 'MEL': 'Melanoma and Nevi', 'NEV': 'Melanoma and Nevi'
    }
    pad_csv = pad_dir / 'pad_ufes_20.csv'
    if pad_csv.exists():
        df_pad = pd.read_csv(pad_csv)
        for _, row in df_pad.iterrows():
            img_id_orig = row['img_id']
            img_path = None
            for folder in pad_dir.iterdir():
                if folder.is_dir() and folder.name.startswith('images_'):
                    p = folder / img_id_orig
                    if p.exists():
                        img_path = p
                        break
            
            if img_path:
                img_id = f'PAD_{p_count:07d}'
                new_path = img_out / f'{img_id}{img_path.suffix}'
                shutil.copy2(img_path, new_path)
                
                label = pad_map.get(row['diagnostic'], 'Systemic Diseases')
                age = row['age'] if pd.notna(row['age']) else 45
                sex = str(row['gender']).lower() if pd.notna(row['gender']) else None
                if sex not in ['male', 'female']: sex = None
                
                skin_map = {'1.0': 'I', '2.0': 'II', '3.0': 'III', '4.0': 'IV', '5.0': 'V', '6.0': 'VI'}
                skin_type = skin_map.get(str(row['fitzpatrick']), 'III')
                
                symptom = 'none'
                if row.get('itch') == 'True' or row.get('itch') == True: symptom = 'itch'
                elif row.get('bleed') == 'True' or row.get('bleed') == True: symptom = 'bleeding'
                elif row.get('hurt') == 'True' or row.get('hurt') == True: symptom = 'pain'
                
                metadata_list.append({
                    'image_id': img_id, 'diagnosis': label, 'age': age, 'sex': sex, 'anatom_site': None, 
                    'duration': 30, 'symptoms': symptom, 'skin_type': skin_type, 'family_history': 'no'
                })
                p_count += 1
print(f"Da xu ly {p_count} anh PAD-UFES-20.")

print("--- GHI FILE METADATA.CSV CHUNG ---")
df_all = pd.DataFrame(metadata_list)
df_all.to_csv('data/raw/metadata.csv', index=False)
print(f"Hoan tat! Tong cong {len(df_all)} anh + metadata.csv")


## Bước 4: Cài đặt thư viện

In [ ]:
!pip install -q timm albumentations opencv-python pyyaml tqdm

## Bước 5: Kết nối Google Drive (Lưu kết quả an toàn + Resume Training)

In [ ]:
from google.colab import drive
import yaml, os

drive.mount('/content/drive')

drive_ckpt = '/content/drive/MyDrive/DermaSense_AI_Checkpoints/stage_b'
os.makedirs(drive_ckpt, exist_ok=True)

with open('configs/stage3_train_multimodal.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['training']['checkpoint_dir'] = drive_ckpt
with open('configs/stage3_train_multimodal.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f)

last_ckpt = os.path.join(drive_ckpt, 'last_checkpoint.pt')
if os.path.exists(last_ckpt):
    print('Tim thay checkpoint cu tren Drive! AI se hoc tiep tu cho cu.')
else:
    print('Chua co checkpoint cu. Bat dau huan luyen tu dau.')

print(f'Checkpoint se duoc luu tai: {drive_ckpt}')

## Bước 6: 🧠 KHỞI CHẠY HUẤN LUYỆN
Bấm Play và đợi 2-3 tiếng. Đừng tắt tab!

Nếu bị ngắt giữa chừng, lần sau chỉ cần chạy lại từ **Bước 1** → AI sẽ tự động học tiếp từ chỗ cũ.

In [ ]:
!PYTHONIOENCODING="utf-8" python -m src.pipelines.run_stage3_train_multimodal

## Bước 7: Tải file AI về máy
Hoặc vào Google Drive → `DermaSense_AI_Checkpoints/stage_b` → tải `best_model.pt`.

In [ ]:
from google.colab import files
import os

best_path = os.path.join(drive_ckpt, 'best_model.pt')
if os.path.exists(best_path):
    print(f'Tim thay best_model.pt ({os.path.getsize(best_path)/1e6:.1f} MB). Dang tai ve...')
    files.download(best_path)
else:
    print('Chua co best_model.pt. Buoc 6 chua chay xong.')